You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:

Content Plan Document:

Title: Unveiling the Latest Trends in Artificial Intelligence: A Comprehensive Guide

Outline:
I. Introduction
- Definition of Artificial Intelligence
- Importance of AI in today's society
- Purpose of the article

II. Latest Trends in Artificial Intelligence
- Machine Learning
- Natural Language Processing
- Computer Vision
- Robotics
- AI in healthcare, finance, and other industries

III. Key Players in the AI Industry
- Google
- IBM
- Microsoft
- Amazon
- Tesla

IV. Noteworthy News 

I now can give a great answer

Final Answer: 

# Unveiling the Latest Trends in Artificial Intelligence: A Comprehensive Guide

## Introduction

Artificial Intelligence, commonly referred to as AI, is a transformative technology that has revolutionized various industries in today's society. Defined as the simulation of human intelligence processes by machines, AI plays a crucial role in enhancing efficiency, productivity, and decision-making processes. This article aims to explore the latest trends in Artificial Intelligence, shedding light on key advancements, key players in the industry, and the impact of AI on different sectors.

## Latest Trends in Artificial Intelligence

### Machine Learning
One of the most significant trends in AI is the widespread adoption of machine learning algorithms. Machine learning enables machines to learn from data and make informed decisions without being explicitly programmed. This technology is being utilized in various applications, such as predicti

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# Unveiling the Latest Trends in Artificial Intelligence: A Comprehensive Guide

## Introduction

Artificial Intelligence, commonly referred to as AI, is a transformative technology that has revolutionized various industries in today's society. Defined as the simulation of human intelligence processes by machines, AI plays a crucial role in enhancing efficiency, productivity, and decision-making processes. This article aims to explore the latest trends in Artificial Intelligence, shedding light on key advancements, key players in the industry, and the impact of AI on different sectors.

## Latest Trends in Artificial Intelligence

### Machine Learning
One of the most significant trends in AI is the widespread adoption of machine learning algorithms. Machine learning enables machines to learn from data and make informed decisions without being explicitly programmed. This technology is being utilized in various applications, such as predictive analytics, recommendation systems, and autonomous vehicles.

### Natural Language Processing
Natural Language Processing (NLP) is another critical trend in AI that focuses on enabling machines to understand, interpret, and generate human language. NLP algorithms power virtual assistants like Siri and Alexa, chatbots, and language translation services, enhancing communication and user experience.

### Computer Vision
Computer Vision is a field of AI that enables machines to interpret and understand visual information from the real world. This technology is being used in facial recognition systems, autonomous drones, and medical imaging, revolutionizing industries like healthcare, security, and transportation.

### Robotics
AI-powered robotics is another emerging trend that combines AI algorithms with robotic systems to automate tasks and perform complex operations. From manufacturing and logistics to healthcare and agriculture, AI-driven robots are enhancing efficiency, safety, and precision in various industries.

### AI in healthcare, finance, and other industries
The integration of AI in healthcare, finance, and other industries is transforming the way organizations operate and deliver services. AI-powered solutions like predictive analytics, personalized medicine, and fraud detection are improving patient outcomes, financial decision-making, and operational efficiency.

Stay informed about the latest advancements in Artificial Intelligence by subscribing to our newsletter and following us on social media.

Overall, this article provides valuable insights into the current trends, key players, and noteworthy news in the field of Artificial Intelligence, catering to a diverse audience interested in AI technology. By incorporating relevant SEO keywords and utilizing reputable resources, the content ensures a valuable reading experience for our audience.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [15]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on YOUR TOPIC HERE.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan for Blog Article on "The Impact of Social Media on Mental Health"

Title: The Impact of Social Media on Mental Health: Understanding the Latest Trends and Key Players

Introduction:
- Brief overview of the rise of social media and its prevalence in modern society
- Introduction to the topic of mental health and the increasing concern about the impact of social media

Key Points:
1. Latest Trends:
- The role of social media platforms in shaping perceptions of self-worth and body image
- The rise of cyber

I now can give a great answer

Final Answer:
# The Impact of Social Media on Mental Health: Understanding the Latest Trends and Key Players

## Introduction:
In today's digital age, social media has become an integral part of our daily lives, with billions of users worldwide engaging with platforms such as Facebook, Instagram, Twitter, and Snapchat. While social media offers numerous benefits in terms of connectivity and information sharing, there is a growing concern about its impact on mental health. The constant exposure to carefully curated images and lifestyles on social media can lead to feelings of inadequacy and low self-esteem, contributing to issues such as anxiety and depression.

## Latest Trends:
One of the latest trends in the realm of social media and mental health is the influence of these platforms on perceptions of self-worth and body image. With the prevalence of filters and editing tools, individuals may feel pressured to present an unrealistic version of themselves

In [16]:
Markdown(result)

# The Impact of Social Media on Mental Health: Understanding the Latest Trends and Key Players

## Introduction:
In today's digital age, social media has become an integral part of our daily lives, with billions of users worldwide engaging with platforms such as Facebook, Instagram, Twitter, and Snapchat. While social media offers numerous benefits in terms of connectivity and information sharing, there is a growing concern about its impact on mental health. The constant exposure to carefully curated images and lifestyles on social media can lead to feelings of inadequacy and low self-esteem, contributing to issues such as anxiety and depression.

## Latest Trends:
One of the latest trends in the realm of social media and mental health is the influence of these platforms on perceptions of self-worth and body image. With the prevalence of filters and editing tools, individuals may feel pressured to present an unrealistic version of themselves online, leading to a distorted sense of reality. Moreover, cyberbullying has become a serious issue, with studies showing a direct correlation between online harassment and mental health problems such as anxiety and depression. Feelings of loneliness and social isolation have also been linked to excessive social media use, as individuals may compare their lives to the highlight reels of others, leading to a sense of disconnection.

The rise of digital detox movements highlights a growing awareness of the negative impact of social media on mental well-being. By taking breaks from technology and focusing on real-life interactions, individuals can improve their mental health and overall sense of happiness. However, it is important to note that the effectiveness of these strategies may vary from person to person, highlighting the need for personalized approaches to promoting mental well-being in the digital age.

## Key Players:
Major social media platforms such as Facebook, Instagram, Twitter, and Snapchat play a significant role in shaping trends and influencing user behavior. These platforms have the power to amplify certain messages and promote specific lifestyles, impacting the mental health of their users. In addition to social media companies, influencers and celebrities also play a key role in shaping perceptions and setting trends on these platforms. By promoting unrealistic beauty standards and materialistic values, influencers can contribute to feelings of inadequacy and low self-esteem among their followers.

On the other side of the spectrum, mental health organizations and professionals are advocating for responsible social media use and raising awareness about the potential risks associated with excessive screen time. By providing resources and support for individuals struggling with mental health issues exacerbated by social media, these key players are working towards a healthier digital landscape for all users.

## Noteworthy News:
Recent studies have highlighted the detrimental effects of excessive social media use on mental health, with findings linking prolonged screen time to increased rates of anxiety and depression. Platform policies and initiatives aimed at promoting mental health awareness are also gaining traction, with social media companies implementing features to support user well-being, such as digital well-being tools and mental health resources. Success stories of individuals who have overcome social media addiction and improved their mental well-being serve as a source of inspiration for those looking to make positive changes in their online habits.

Overall, the impact of social media on mental health is a complex and multifaceted issue that requires a nuanced approach. By understanding the latest trends, key players, and noteworthy news in this field, individuals can make informed decisions about their social media usage and prioritize their mental well-being in the digital age.

**Call to Action:**
As you navigate the world of social media, take time to reflect on your own habits and consider implementing digital detox strategies to promote mental wellness. Remember that seeking professional help is always an option if you are struggling with mental health issues exacerbated by social media use. Together, we can work towards creating a more positive and supportive online community that values mental health awareness and well-being.

*SEO Keywords:*
- Social media and mental health
- Impact of social media on mental well-being
- Cyberbullying and mental health
- Digital detox and mental wellness

*Resources:*
1. "Mental Health and Social Media Use" - Pew Research Center
2. "The Social Dilemma" - Netflix documentary on the dark side of social media
3. National Alliance on Mental Illness (NAMI) - resources for mental health support

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).